# EP01 验证样片（51 秒，免费 T4，约 6~10 分钟跑完全流程）

**只需上传 1 个文件：`llm_gpu_bundle.zip`**（拖进左侧文件面板 /content/）

验证内容：GPU 渲染（前 3 场景 1530 帧）→ 配乐 → 编码 → **F5-TTS 克隆云健男声配音**（2 句）→ 混音出片。
这是完整管线的浓缩版——样片满意后用 `colab_t4_test.ipynb` 跑整集。

菜单「代码执行程序 → 更改代码执行程序类型」→ **T4 GPU**

In [ ]:
# 1) GPU 检查 + 挂载 Drive（授权弹窗点允许）
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无！菜单里切换 T4')
assert os.path.exists('/content/llm_gpu_bundle.zip'), '请先上传 llm_gpu_bundle.zip 到 /content/'
from google.colab import drive
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/ColabOutputs', exist_ok=True)
print('Drive 就绪')

In [ ]:
# 2) 解包 + 依赖（含 F5-TTS；首次格 5 会再下载 1.35GB 模型）
%%bash
cd /content && rm -rf work && mkdir work && cd work
unzip -q ../llm_gpu_bundle.zip
pip install -q edge-tts f5-tts 2>&1 | tail -1
echo '--- 解包完成 ---'
ls /content/work/ref_voice.wav && echo '内置云健男声就绪 → F5 克隆配音'

In [ ]:
# 3) GPU 渲染 51s = 1530 帧（T4 预计 1~3 分钟）
%cd /content/work/gpu
import os, time, torch
from PIL import Image
import truntime as T

T.set_device('cuda')
mod, shots, total = T.load_episode('ep01_demo')
print(f"[{total}s] 场景: {[s['name'] for s in shots]}")
os.makedirs('frames_ep01_demo', exist_ok=True)
tf = int(total * 30)
t0 = time.time()
for i in range(tf):
    arr = T.render_by_time(mod, shots, min(i/30, total-0.1))
    Image.fromarray(arr).save(f'frames_ep01_demo/f{i:05d}.jpg', quality=92, subsampling=0)
    if i % 500 == 0:
        torch.cuda.synchronize(); print(f'{i}/{tf} {time.time()-t0:.0f}s', flush=True)
torch.cuda.synchronize()
fps = tf / (time.time() - t0)
print(f'渲染完成: {tf} 帧 @ {fps:.1f} fps —— 按此速度整集 10800 帧约 {10800/fps/60:.1f} 分钟')

In [ ]:
# 4) 配乐（纯 numpy，秒级）+ 无声段编码（约 1 分钟）
%%bash
cd /content/work/gpu
python - <<'PY'
import sys; sys.path.insert(0, '..')
import truntime as T
import make_music as MM
mod, shots, total = T.load_episode('ep01_demo')
MM.render_soundtrack(mod, shots)
print('配乐完成')
PY
cd /content/work
mv gpu/soundtrack_ep01_demo.wav .
mkdir -p out
ffmpeg -y -loglevel error -framerate 30 -i gpu/frames_ep01_demo/f%05d.jpg -i soundtrack_ep01_demo.wav \n  -c:v libx264 -preset medium -crf 17 -pix_fmt yuv420p -c:a aac -b:a 256k \n  -movflags +faststart out/ep01_demo.mp4
ffprobe -v error -show_entries format=duration -of default=noprint_wrappers=1:nokey=1 out/ep01_demo.mp4
echo '无声样片 out/ep01_demo.mp4 完成'

In [ ]:
# 5) F5-TTS 克隆云健男声配音（首跑下模型 1.35GB；2 句约 1 分钟）→ 混音 → 成品
%%bash
cd /content/work
python narrate.py ep01_demo
ls -la out/
cp out/ep01_demo_narrated.mp4 /content/drive/MyDrive/ColabOutputs/
echo '==========================================='
echo '验证样片完成 → Drive/ColabOutputs/ep01_demo_narrated.mp4'
echo '（也可在左侧 /content/work/out/ 右键直接下载）'
echo '==========================================='

## 看什么
- 画面：暗角/辉光/转场与原版观感一致（GPU 移植已做过 24 帧 PSNR 校验）
- 配音：两句解说（约 17s、38.5s 处），听音色是否贴近原系列云健声、背景乐是否自动闪避
- 格 3 打印的 fps：决定整集渲染时长预估

## 样片满意后
用 `colab_t4_test.ipynb` 跑整集（约 20~30 分钟）；配音想换自己的声音，上传 `ref_voice.wav` 即自动覆盖